# PANDAS 02
Practica mas avanzada de Pandas

In [ ]:
import pandas as pd 
df = pd.read_csv('personas.csv',delimiter=';')
df

Podemos revisar los primeros 5 y ultimos 5 registros para tener una idea de la data

In [ ]:
df.head()

In [ ]:
df.tail()

## renombrado de columnas
Podemos renombrar columnas del dataframe usando rename.
Para eso, pasamos un DICCIONARIO que tiene key=nombre_viejo , value=nombre_nuevo

In [ ]:
df_con_renombres = df.rename(columns={"Ocupacion":"Trabajo","Salario":"Ingresos"})
df_con_renombres.head()
# si quisieramos editar el mismo dataframe en si, pondiamos [ df = df.rename(... ]

## Filtrado de columnas
Podemos filtrar los elementos de un dataframe.
Para ello, dentro de la definicion de dataframe vamos a tener que generar una SERIE de True/Falses (una Serie de tipo Booleano) que nos indiquen que filas se van a tomar y cuales no.... Las filas con True se toman mientras que las que den False no se toman.

En un ejemplo paso a paso esto quedaria asi:


In [ ]:
# recordando la estructura original:
df.head(3)

In [ ]:
# Quiero filtar todos aquellos registros con edades mayores a 45
# Lo voy viendo paso a paso
# Tengo esta Serie que es mi columna edad:
serie_edades = df["Edad"]
serie_edades.head(5)

In [ ]:
# puedo generar una serie de booleanos pasandole una condicion
serie_booleanos_condicion = serie_edades >= 45  # df["Edad"] >= 45 
serie_booleanos_condicion.head(5)

In [ ]:
# y si ahora, uso esa serie de booleanos en mi df, obtengo el df filtrado
df_filtrado = df[serie_booleanos_condicion]
df_filtrado.head(5)

In [ ]:
# puedo hacer todo esto de forma mas visual directamente asi:
# df_filtrado = df[ (serie de booleanos generada por una condicion) ]
df_filtrado   = df[ df['Edad'] >= 45 ]
df_filtrado.head(5)

mas ejemplos de filtros posibles

In [ ]:
# condiciones multiples
# ejs: medicos que ganen mas de 2100
df_filtrado = df[( (df['Ocupacion'] == 'Medico') | (df['Ocupacion'] == 'Medica') ) & ( df['Salario'] > 1800 ) ]
df_filtrado
# (notar como no aparece el registro 9 de arriba, mas adelante veremos como hacer limpieza de datos)

In [ ]:
# aparte del OR (|) y del AND (&) tambien tengo el NOT ~
# Ejemplo anterior pero inviertiendo condicion de salario para que NO me traiga los mayores a 1800
df_filtrado2 = df[( (df['Ocupacion'] == 'Medico') | (df['Ocupacion'] == 'Medica') ) & ~ ( df['Salario'] > 1800 ) ]
df_filtrado2


In [ ]:
# filtrando como si fuera con sql
# OJO: los operadores OR, AND van en MINUSCULAS
df_filtrado = df.query(" ((Ocupacion == 'Medico') or ( Ocupacion == 'Medica')) and Salario > 1800 ")
df_filtrado

In [ ]:
# o sino 
df_filtrado = df.query(" Ocupacion in ['Medico' ,'Medica'] and Salario > 1800 ")
df_filtrado

In [ ]:
# filtrando casos que empiencen con la letra A
df_filtrado = df[df["Nombre"].str.startswith('A')]
df_filtrado.head(3)

In [ ]:
# solo asi puedo hacer un 'RLIKE'
df_filtrado = df[ df["Nombre"].str.contains('^a.*o$', case=False, regex=True, na=False) ]
df_filtrado.head(3)

## Funciones de agregacion
Tenemos funciones que aceptan una serie de input y retornan un solo valor de output.

In [ ]:
# recordando mi df
df.head()

In [ ]:
# Quiero obtener, de el campo 'Salario' el promedio, el valor minimo y el valor maximo.
print( "Promedio:" , df['Salario'].mean() )
print( "Minimo:" , df['Salario'].min() )
print( "Maximo:" , df['Salario'].max() )

Dada una serie como 'Salario', estos mismos datos los puedo obtener con 'Describe'

In [ ]:
df['Salario'].describe()

Tambien podemos usar el metodo para varias columnas numericas a la vez

In [ ]:
df[['Salario','Edad']].describe()

In [ ]:
# redondeando cantidad de decimales
df[['Salario','Edad']].describe().round(2)

## Group By
Podemos agrupar las columnas como cuando queremos ver resultados agregados en SQL

In [ ]:
# recordando mi df
df.head()

In [ ]:
# esto devuelve un objeto de tipo DataFrameGroupBy.
df.groupby(['Ciudad'])
# no es un dataframe, es un objeto que tiene metodos para poder agrupar y hacer operaciones sobre el df original.

In [ ]:
# aca nos muestra, por ejemplo, todas las ciudades y la cantidad de registros que hay en cada una.
df.groupby(['Ciudad']).count()

In [ ]:
# Para hacer un 'select count(x)' de una sola columna
df.groupby(['Ciudad'])[['Nombre']].count()

In [ ]:
# esta es otra forma de hacerlo, pero tiene el invonveniente de que calcula el conteo para TODOS los campos y hace el filtro despues, mientras que la forma anterior solo calcula el conteo de la columna que nos interesa.
df.groupby(['Ciudad']).count()[['Nombre']]

In [ ]:
# para MAS de una columna
df.groupby(['Ciudad'])[['Nombre','Salario']].count()

Es interesante ver como en la funcion 'group by' usa las ciudades como indices

In [ ]:
df_agrupado = df.groupby(['Ciudad'])[['Nombre','Salario']].count() 
df_agrupado.head()

lo cual quiere decir que ahora 'ciudad' NO es mas una columna del df 

In [ ]:
df_agrupado.columns

para poder volver a tener indices numericos y que 'Ciudad' pase a ser una columna nuevamente, se puede usar 'reset_index'

In [ ]:
df_agrupado = df_agrupado.reset_index()  # para que la columna 'Ciudad' deje de ser el indice y pase a ser una columna mas del df
df_agrupado

## Sorting

ordenando dataframes segun su valor de columna

In [ ]:
df.sort_values(['Salario'],ascending=False).head(5)

## NaN values
Manejo de valores nulos

In [ ]:
# revision rapida general de campos con valores NULL
df.isna().sum()

In [ ]:
# voy a ver los registros que tengan campo 'SALARIO' en NULL
nan_df_salario = df[ df['Salario'].isna() ]
nan_df_salario.head(5)

In [ ]:
# podemos dropear los registros con NULLs si queremos 
# para dropear TODO lo que tenga algun Null:
df_without_nan = df.dropna() 
df_without_nan.head(3)

In [ ]:
# para dropear solo los nulls de Salario
df_without_nan = df.dropna(subset=['Salario']) 
df_without_nan[30:37] # porcion registros sin nulls de Salario 

In [ ]:
# Nota: previo a modificacion, ver que los tipos de dato de puntaje desempenio y salario son de tipo numerico
df.dtypes

Tambien podemos reemplazar los valores nulos por una cadena de texto 

In [ ]:
df_v2 = df.fillna('[VALOR_NULL]')
df_v2[['Salario','Puntaje_Desempenio']][30:37]

In [ ]:
# pasan a ser de tipo 'object' 
df_v2.dtypes

In [ ]:
# podria haberlos reemplazado con un valor numerico por ejemplo: 
df_v2 = df.fillna(0)
df_v2[['Salario','Puntaje_Desempenio']][30:37]

In [ ]:
df_v2.dtypes

Tambien se pueden rellenar con distintas cadenas/valores para distintos campos usando un diccionario, de la siguiente manera:

In [ ]:
df_v2 = df.fillna(
        {
            'Salario':'[SALARIO_NULL]',
            'Puntaje_Desempenio':'[PUNTAJE_NULL]'
        }
    )
df_v2[['Salario','Puntaje_Desempenio']][30:37]

## Write CSVs 
Podemos escribir un CSV a partir de un dataframe

In [ ]:
# le agrego una pequeña porcion de codigo para que imprima si el archivo ya existia previamente o no 

from pathlib import Path

NOMBRE_ARCHIVO = 'dataframe_to_csv.csv'
ruta = Path(NOMBRE_ARCHIVO)

if ruta.is_file():
    print(f"Reemplazando archivo existente '{NOMBRE_ARCHIVO}' ")
else:
    print(f"Creando archivo nuevo '{NOMBRE_ARCHIVO}'")

df_ejemplo = df_v2[['Salario','Puntaje_Desempenio']][30:37]
df_ejemplo.to_csv( NOMBRE_ARCHIVO ,index=False) # para que no me guarde el indice como una columna mas del CSV
